# FashionCLIP Embedding Generation

This notebook generates visual embeddings for your fashion items using FashionCLIP.

**Instructions:**
1. Upload your `items.csv` file when prompted
2. Run all cells
3. Download `embeddings.npy` and `item_ids.npy` when complete
4. Place them in `data/embeddings/` folder

In [ ]:
# Install dependencies
!pip install -q transformers torch pillow pandas httpx tqdm

In [ ]:
import os
import re
import io
from typing import List, Optional
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
from PIL import Image
import httpx
from tqdm.auto import tqdm
from google.colab import files

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Upload your items.csv file
print("Please upload your items.csv file:")
uploaded = files.upload()
items_filename = list(uploaded.keys())[0]
print(f"Uploaded: {items_filename}")

In [ ]:
# Load and parse items
df = pd.read_csv(items_filename)
print(f"Loaded {len(df)} items")
print(f"Columns: {df.columns.tolist()}")

# Parse image URLs from the array format
def parse_image_urls(urls_str):
    """Parse image URLs from string format like ["url1" "url2"]"""
    if pd.isna(urls_str) or not urls_str:
        return []
    urls_str = str(urls_str).strip()
    if urls_str.startswith('[') and urls_str.endswith(']'):
        urls_str = urls_str[1:-1]
    url_pattern = r'https?://[^\s\"\']+'  
    return re.findall(url_pattern, urls_str)

# Extract primary image URL
df['parsed_urls'] = df['image_urls'].apply(parse_image_urls)
df['primary_image'] = df['parsed_urls'].apply(lambda x: x[0] if x else None)
df['num_images'] = df['parsed_urls'].apply(len)

# Filter to items with valid URLs
valid_df = df[df['primary_image'].notna()].copy()
print(f"Items with valid image URLs: {len(valid_df)}")
print(f"\nSample URLs:")
for url in valid_df['primary_image'].head(3):
    print(f"  {url[:80]}...")

In [ ]:
# Load FashionCLIP model
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification

print("Loading FashionCLIP model...")
model_name = "patrickjohncyh/fashion-clip"

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForZeroShotImageClassification.from_pretrained(
    model_name,
    torch_dtype=torch.float32
).to(device)
model.eval()

print(f"Model loaded on {device}")

# Test with a simple image
test_img = Image.new('RGB', (224, 224), color='red')
test_inputs = processor(images=test_img, return_tensors="pt")
test_inputs = {k: v.to(device) for k, v in test_inputs.items()}

with torch.no_grad():
    # Access the underlying CLIP model for features
    test_output = model.model.get_image_features(**test_inputs)

if torch.isnan(test_output).any():
    print("WARNING: Model produces NaN!")
else:
    print(f"Model test passed! Output shape: {test_output.shape}")

In [ ]:
# Image download helper
def download_image(url: str, timeout: float = 15.0) -> Optional[Image.Image]:
    """Download image from URL."""
    try:
        with httpx.Client(timeout=timeout, follow_redirects=True) as client:
            response = client.get(url)
            response.raise_for_status()
            image = Image.open(io.BytesIO(response.content))
            return image.convert('RGB')
    except Exception as e:
        return None

def download_images_parallel(urls: List[str], max_workers: int = 8) -> List[Optional[Image.Image]]:
    """Download multiple images in parallel."""
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        images = list(executor.map(download_image, urls))
    return images

# Test download
test_url = valid_df['primary_image'].iloc[0]
test_download = download_image(test_url)
if test_download:
    print(f"Test download successful! Image size: {test_download.size}")
    display(test_download.resize((200, 200)))
else:
    print("Test download failed!")

In [ ]:
# Generate embeddings
@torch.no_grad()
def generate_embeddings(
    image_urls: List[str],
    item_ids: List[int],
    batch_size: int = 32,
    max_workers: int = 8,
) -> tuple:
    """Generate embeddings for all images."""
    
    all_embeddings = []
    all_item_ids = []
    failed_count = 0
    
    for batch_start in tqdm(range(0, len(image_urls), batch_size), desc="Generating embeddings"):
        batch_urls = image_urls[batch_start:batch_start + batch_size]
        batch_ids = item_ids[batch_start:batch_start + batch_size]
        
        # Download images
        batch_images = download_images_parallel(batch_urls, max_workers=max_workers)
        
        # Filter successful downloads
        valid_images = []
        valid_ids = []
        for img, item_id in zip(batch_images, batch_ids):
            if img is not None:
                valid_images.append(img)
                valid_ids.append(item_id)
            else:
                failed_count += 1
        
        if not valid_images:
            continue
        
        # Process batch
        inputs = processor(images=valid_images, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Get embeddings from underlying CLIP model
        batch_embeddings = model.model.get_image_features(**inputs)
        
        # Check for NaN
        if torch.isnan(batch_embeddings).any():
            print(f"Warning: NaN in batch {batch_start}, replacing with zeros")
            batch_embeddings = torch.nan_to_num(batch_embeddings, nan=0.0)
        
        # Normalize
        norms = batch_embeddings.norm(dim=-1, keepdim=True)
        norms = torch.clamp(norms, min=1e-8)
        batch_embeddings = batch_embeddings / norms
        
        # Store
        all_embeddings.append(batch_embeddings.cpu().numpy())
        all_item_ids.extend(valid_ids)
    
    embeddings_array = np.vstack(all_embeddings) if all_embeddings else np.zeros((0, 512))
    
    print(f"\nGenerated {len(embeddings_array)} embeddings")
    print(f"Failed downloads: {failed_count}")
    
    return embeddings_array, all_item_ids

# Generate embeddings
image_urls = valid_df['primary_image'].tolist()
item_ids = valid_df['item_id'].tolist()

print(f"Processing {len(image_urls)} images...")
embeddings, successful_ids = generate_embeddings(
    image_urls=image_urls,
    item_ids=item_ids,
    batch_size=32,
    max_workers=8,
)

In [ ]:
# Validate embeddings
print(f"Embeddings shape: {embeddings.shape}")
print(f"Number of item IDs: {len(successful_ids)}")

# Check for NaN
nan_count = np.isnan(embeddings).sum()
print(f"NaN values: {nan_count}")

# Check for zero vectors
zero_vectors = np.all(embeddings == 0, axis=1).sum()
print(f"Zero vectors: {zero_vectors}")

# Sample statistics
print(f"\nEmbedding statistics:")
print(f"  Mean: {embeddings.mean():.4f}")
print(f"  Std: {embeddings.std():.4f}")
print(f"  Min: {embeddings.min():.4f}")
print(f"  Max: {embeddings.max():.4f}")

In [ ]:
# Save embeddings
np.save('embeddings.npy', embeddings)
np.save('item_ids.npy', np.array(successful_ids))

print("Saved embeddings.npy and item_ids.npy")
print(f"\nFile sizes:")
print(f"  embeddings.npy: {os.path.getsize('embeddings.npy') / 1024 / 1024:.2f} MB")
print(f"  item_ids.npy: {os.path.getsize('item_ids.npy') / 1024:.2f} KB")

In [ ]:
# Download files
print("Downloading embeddings.npy...")
files.download('embeddings.npy')

print("Downloading item_ids.npy...")
files.download('item_ids.npy')

print("\n" + "="*50)
print("DONE!")
print("="*50)
print("\nPlace these files in your local data/embeddings/ folder:")
print("  - embeddings.npy")
print("  - item_ids.npy")
print("\nThen run training locally:")
print("  python scripts/train.py --config configs/config.yaml --epochs 50")